# 14 — The Gaussian Geodesic

Follow the shortest path between two Gaussians and see that it never leaves the Gaussian family: every point of the path is a Gaussian, whose mean slides along a line and whose covariance ellipse turns and rescales. This is the place where the optimal path between continuous distributions in the plane, and not only the distance, can be written down in full.

**Prerequisites:** `03/06_1d_quantile_closed_form` (quantile matching on a line), `03/07_mccann_geodesic` (mass slides along straight tracks), `03/12_bures_wasserstein` (the closed-form $W_2$ between Gaussians), `03/13_the_bures_bridge` (the matrix term is the Bures distance, and the Uhlmann fidelity $F_U$).

**What you'll be able to do**
- Work out by hand the optimal map between two one-dimensional Gaussians and see that the path interpolates the standard deviation.
- Build the symmetric matrix $A$ that carries one covariance onto another, state why Brenier's theorem singles it out, and check it against a rival map.
- Trace the $W_2$ geodesic between two Gaussians in the plane and check it against a rival path by its length.
- Explain why the geodesic formula, fed density matrices, leaves the set of quantum states, and add four rows to the dictionary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import to_rgb

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.transport.gaussian import (
    bures_matrix_distance,
    bures_wasserstein_distance,
    gaussian_geodesic,
    gaussian_ot_map,
)

rng = np.random.default_rng(42)   # the points that travel along their tracks (section 3)
viz.use_course_style()

## Where we are

`03/12` gave the distance between two Gaussians, `03/13` showed that its matrix term is the quantum Bures distance, and `03/07` showed on a line that sliding each grain of mass along a straight track is the geodesic of $W_2$, the shortest path. Between Gaussians the optimal map is affine: it stretches the plane along two perpendicular directions and shifts it. An affine map sends a Gaussian to a Gaussian, so every intermediate distribution is Gaussian.

That the path is the shortest is taken from `03/07` here and checked through two signatures: a distance from the start proportional to the time, and a length equal to the distance while a rival path is longer. `03/15` explains why it is shortest.

## 1. One dimension, by hand

Start where everything can be done on paper. Take $\mu_0 = \mathcal{N}(m_0, \sigma_0^2)$ and $\mu_1 = \mathcal{N}(m_1, \sigma_1^2)$ on the line. In `03/06` you matched quantiles. For a Gaussian the quantile at level $u$ is $m + \sigma\,z_u$, with $z_u$ the standard normal quantile, so the matching reads

$$ T(x) = m_1 + \frac{\sigma_1}{\sigma_0}\,(x - m_0), $$

an affine map that recentres, then rescales. Let each point travel along a straight track from $x$ to $T(x)$, as in `03/07`. At time $t$ it has reached

$$ x_t = (1-t)\,x + t\,T(x) = \underbrace{(1-t)\,m_0 + t\,m_1}_{m_t} + \bigl((1-t) + t\,\tfrac{\sigma_1}{\sigma_0}\bigr)\,(x - m_0). $$

Since $x - m_0$ is Gaussian with standard deviation $\sigma_0$, the point $x_t$ is Gaussian with mean $m_t$ and standard deviation $(1-t)\,\sigma_0 + t\,\sigma_1$. The **standard deviation** is interpolated linearly, not the variance, as `03/12` hinted when it called the $(m, \sigma)$ plane flat.

**A worked case.** Move $\mathcal{N}(0, 1)$ onto $\mathcal{N}(4, 0.5^2)$. At $t = \tfrac12$ the mean is $2$ and the standard deviation is $\tfrac12(1 + 0.5) = 0.75$. Averaging the variances instead would give $\sqrt{0.625} = 0.7906$. Only the first is a shortest path (by `03/12`, $W_2$ is Euclidean in the $(m, \sigma)$ plane). The next cell computes the path with `gaussian_geodesic` and sets it beside the mixture, which blends the two densities instead of moving the mass.

In [ ]:
m0, s0, m1, s1 = 0.0, 1.0, 4.0, 0.5            # N(0, 1) -> N(4, 0.5^2)
x = np.linspace(-3.5, 7.0, 600)

def normal_pdf(x, m, s):
    return np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi))

def darker(color, amount=0.35):
    # the palette colour blended toward the text colour, for lines that must stay readable
    return tuple((1 - amount) * c + amount * d for c, d in zip(to_rgb(color), to_rgb(COLORS["text"])))

times = [0.0, 0.25, 0.5, 0.75, 1.0]
path = [gaussian_geodesic([m0], [[s0 ** 2]], [m1], [[s1 ** 2]], t) for t in times]
for t, (m_t, c_t) in zip(times, path):
    print(f"t = {t:.2f}   mean = {m_t[0]:.2f}   std = {np.sqrt(c_t[0, 0]):.4f}")
print(f"midpoint std by interpolating sigma      : {np.sqrt(path[2][1][0, 0]):.4f}")
print(f"square root of the mean variance         : {np.sqrt(0.5 * (s0 ** 2 + s1 ** 2)):.4f}")

cmap = plt.colormaps["viridis"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for k, (t, (m_t, c_t)) in enumerate(zip(times, path)):
    axes[0].plot(x, normal_pdf(x, m_t[0], np.sqrt(c_t[0, 0])), color=cmap(k / (len(times) - 1)), lw=2.2, label=f"t = {t:.2f}")
axes[0].set_title("Moving the mass: the geodesic")
axes[0].set_xlabel("x")
axes[0].set_ylabel("density")
axes[0].legend(loc="upper right")

mid_m, mid_c = path[2]
mixture = 0.5 * normal_pdf(x, m0, s0) + 0.5 * normal_pdf(x, m1, s1)
axes[1].plot(x, normal_pdf(x, m0, s0), color=darker(viz.SOURCE_COLOR), lw=1.8, ls="--", label="start")
axes[1].plot(x, normal_pdf(x, m1, s1), color=darker(viz.TARGET_COLOR), lw=1.8, ls="--", label="end")
axes[1].plot(x, normal_pdf(x, mid_m[0], np.sqrt(mid_c[0, 0])), color=viz.FLOW_COLOR, lw=2.6, label="geodesic, t = 0.5")
axes[1].plot(x, mixture, color=COLORS["highlight"], lw=2.6, label="mixture, t = 0.5")
axes[1].set_title("Mixing the mass: the mixture path")
axes[1].set_xlabel("x")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

**Read the figure.** On the left, five snapshots of the geodesic, from violet to yellow. Each curve is a single bump that travels to the right and narrows, so it grows taller. On the right, the geodesic midpoint (green) sits beside the mixture midpoint (rose) of the two ends. The mixture has two humps, one over each end: it has moved no mass, only faded one bump out and the other in. The geodesic midpoint is a single bump centred halfway, the contrast of `03/07` between sliding and mixing mass. A mixture of two different Gaussians is never Gaussian, whereas the moved mass stays Gaussian.

## 2. In higher dimension, the matrix map

In $\mathbb{R}^d$ the same idea needs a matrix in place of the ratio $\sigma_1/\sigma_0$. Take $\mu_0 = \mathcal{N}(m_0, \Sigma_0)$ and $\mu_1 = \mathcal{N}(m_1, \Sigma_1)$, with both covariances positive definite (invertible), so that $\Sigma_0^{-1/2}$ exists. We look for an affine map $T(x) = m_1 + A\,(x - m_0)$. If $x \sim \mu_0$, then $T(x)$ is Gaussian with mean $m_1$ and covariance $A\,\Sigma_0\,A^\top$, so $T$ pushes $\mu_0$ onto $\mu_1$ exactly when

$$ A\,\Sigma_0\,A^\top = \Sigma_1. $$

That equation has many solutions: for any rotation $R$, the matrix $B = \Sigma_1^{1/2}\,R\,\Sigma_0^{-1/2}$ is one, because $B\,\Sigma_0\,B^\top = \Sigma_1^{1/2}\,R\,R^\top\,\Sigma_1^{1/2} = \Sigma_1$. They do not move the mass the same distance. Which one is optimal?

**Brenier's theorem.** The course has not stated the result that answers this; here it is, cited and not proved (Brenier, 1991). When $\mu_0$ has a density and both distributions have finite second moments, the optimal transport problem for the squared distance has a unique solution, and it is a map $T = \nabla\varphi$, the gradient of a convex function $\varphi$. Conversely, a map that is the gradient of a convex function and pushes $\mu_0$ onto $\mu_1$ is the optimal one. A Gaussian with invertible covariance has a density with respect to Lebesgue measure and all its moments, so the theorem applies to our pair. It does not apply as stated when $\Sigma_0$ is singular, since then $\mu_0$ has no density.

A linear map $x \mapsto Bx$ is the gradient of a convex quadratic exactly when $B$ is symmetric and positive semi-definite. So we look for the **symmetric positive** solution of $A\,\Sigma_0\,A = \Sigma_1$, and it can be found on paper. Write $S_0 = \Sigma_0^{1/2}$. Multiplying the equation by $S_0$ on both sides gives $(S_0\,A\,S_0)^2 = S_0\,\Sigma_1\,S_0$. The matrix $S_0 A S_0$ is positive, and a symmetric positive definite matrix has a unique positive square root, so $S_0 A S_0 = (S_0\,\Sigma_1\,S_0)^{1/2}$ and

$$ A = \Sigma_0^{-1/2}\bigl(\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}\bigr)^{1/2}\Sigma_0^{-1/2}. $$

This is the formula behind `gaussian_ot_map`; Bhatia, Jain and Lim (2019, Section 3) write it as the geometric mean $\Sigma_0^{-1}\#\Sigma_1$.

**The diagonal case.** When both covariances are diagonal the formula collapses to $A_{ii} = \sqrt{\Sigma_{1,ii}/\Sigma_{0,ii}}$: each axis is a one-dimensional problem. For $\Sigma_0 = \mathrm{diag}(1, 4)$ and $\Sigma_1 = \mathrm{diag}(9, 1)$ this gives $A = \mathrm{diag}(3, \tfrac12)$: the first axis is stretched threefold and the second halved. The next cell confirms it, then puts a rival map beside $A$ for the pair used in the rest of the notebook.

In [ ]:
# By hand: diag(1, 4) -> diag(9, 1) gives diag(3, 0.5).
A_diag = gaussian_ot_map(np.diag([1.0, 4.0]), np.diag([9.0, 1.0]))
print("diagonal case, A =")
print(np.round(A_diag, 4))

# The pair used in the rest of the notebook (a pair that does not commute).
mean_0 = np.array([-2.5, -1.0])
cov_0 = np.array([[1.5, 0.4], [0.4, 0.6]])
mean_1 = np.array([2.5, 1.5])
cov_1 = np.array([[0.5, -0.3], [-0.3, 1.4]])

A = gaussian_ot_map(cov_0, cov_1)
print(f"\nA Sigma_0 A == Sigma_1 ?     {bool(np.allclose(A @ cov_0 @ A.T, cov_1, atol=1e-9))}")
print(f"A symmetric?                 {bool(np.allclose(A, A.T))}")
print(f"eigenvalues of A             {np.round(np.linalg.eigvalsh(A), 4)}   (all positive)")

# A rival: B = Sigma_1^{1/2} R Sigma_0^{-1/2} also satisfies B Sigma_0 B^T = Sigma_1, for any rotation R.
def sqrtm_psd(M):
    vals, vecs = np.linalg.eigh(M)
    return (vecs * np.sqrt(vals)) @ vecs.T

S1, S0_inv = sqrtm_psd(cov_1), np.linalg.inv(sqrtm_psd(cov_0))

def transport_cost(B):
    # mean squared displacement E|x - B x|^2 for x ~ N(0, cov_0): tr((I - B) Sigma_0 (I - B)^T)
    D = np.eye(2) - B
    return float(np.trace(D @ cov_0 @ D.T))

print(f"\ncost of the symmetric map A                    {transport_cost(A):.4f}"
      f"   (squared Bures matrix distance: {bures_matrix_distance(cov_0, cov_1):.4f})")
for angle in [0.0, 0.3, 0.5]:
    R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    B = S1 @ R @ S0_inv
    ok = bool(np.allclose(B @ cov_0 @ B.T, cov_1, atol=1e-9))
    print(f"rival, rotation by {angle:.1f} rad: pushes Sigma_0 onto Sigma_1 = {ok},  cost {transport_cost(B):.4f},"
          f"  symmetric = {bool(np.allclose(B, B.T))}")

**Read the output.** The diagonal map is $\mathrm{diag}(3, 0.5)$, as predicted on paper. For the notebook's pair, $A$ reproduces $\Sigma_1$, is symmetric, and has positive eigenvalues $0.4987$ and $1.8207$: a pure stretch, with no rotation, along two perpendicular directions. Its cost, the mean squared displacement of a point, is $0.7175$. That equals the squared Bures matrix distance of `03/12` (the squared distance of the closed form of Dowson and Landau, 1982, and Olkin and Pukelsheim, 1982, once the means are removed), so the optimal map pays exactly the distance. It is a squared quantity: the distance itself, $0.8470$ in the next section, is its square root. The rival with rotation $0$ is a near tie, $0.7179$ against $0.7175$: a map can be almost optimal without being the optimum. At $0.5$ radian the cost is $1.0959$, more than half as much again, so the rotation makes the points travel farther on average.

## 3. The geodesic: straight tracks, a Gaussian at every time

With the map in hand the path follows as in one dimension. Each point travels along a straight track from $x$ to $T(x) = m_1 + A\,(x - m_0)$, so at time $t$ it sits at

$$ x_t = (1-t)\,x + t\,T(x) = m_t + M_t\,(x - m_0), \qquad m_t = (1-t)\,m_0 + t\,m_1, \quad M_t = (1-t)\,I + t\,A. $$

This is an affine image of a Gaussian, so the law at time $t$ is Gaussian:

$$ \mu_t = \mathcal{N}\big(m_t,\ M_t\,\Sigma_0\,M_t^\top\big). $$

The mean moves along a straight line and the covariance is carried by $M_t$, an average of $I$ and $A$. The stretching directions of $A$ are not those of $\Sigma_0$, so the ellipse turns as well as growing or shrinking. This is the displacement interpolation of McCann (1997) and `03/07`, in matrix form; Bhatia, Jain and Lim (2019, Section 4, formula (39)) give the same geodesic for covariance matrices, and `gaussian_geodesic` implements it.

The next cell draws five ellipses of the path and follows fourteen points drawn from the starting Gaussian along their straight tracks. It checks the covariance formula by sampling: two hundred thousand points moved halfway along their tracks should have the covariance $M_{1/2}\,\Sigma_0\,M_{1/2}$. The law is Gaussian because the map is affine, so only second moments need checking. It also counts the pairs of tracks that cross.

In [ ]:
def ellipse(mean, cov, n_std=1.5, **kwargs):
    # A covariance ellipse at n_std standard deviations.
    vals, vecs = np.linalg.eigh(0.5 * (cov + cov.T))
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = float(np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0])))
    width, height = (2.0 * n_std * np.sqrt(np.clip(vals, 0.0, None))).tolist()
    return patches.Ellipse(xy=tuple(mean), width=width, height=height, angle=angle, **kwargs)

ts = np.linspace(0.0, 1.0, 5)
snapshots = [gaussian_geodesic(mean_0, cov_0, mean_1, cov_1, t) for t in ts]
means = np.array([gaussian_geodesic(mean_0, cov_0, mean_1, cov_1, t)[0] for t in np.linspace(0, 1, 50)])

# Fourteen points drawn from the start, and their straight tracks to T(x) = m1 + A (x - m0).
points = rng.multivariate_normal(mean_0, cov_0, size=14)
destinations = mean_1 + (points - mean_0) @ A.T

# Check the covariance formula by sampling: many points moved to t = 0.5 along their tracks.
many = rng.multivariate_normal(mean_0, cov_0, size=200_000)
halfway = 0.5 * many + 0.5 * (mean_1 + (many - mean_0) @ A.T)
_, cov_half = gaussian_geodesic(mean_0, cov_0, mean_1, cov_1, 0.5)
print("covariance of 200,000 points moved to t = 0.5:")
print(np.round(np.cov(halfway.T), 3))
print("covariance given by the geodesic formula:")
print(np.round(cov_half, 3))
print(f"largest entry-wise difference: {np.abs(np.cov(halfway.T) - cov_half).max():.4f}")

# Count the pairs of drawn tracks that cross (segment intersection test).
def segments_cross(p1, p2, q1, q2):
    def orient(a, b, c):
        return np.sign((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))
    return orient(p1, p2, q1) != orient(p1, p2, q2) and orient(q1, q2, p1) != orient(q1, q2, p2)

crossings = sum(segments_cross(points[i], destinations[i], points[j], destinations[j])
                for i in range(14) for j in range(i + 1, 14))
print(f"pairs of tracks that cross, out of {14 * 13 // 2}: {crossings}")

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.colormaps["viridis"]
for k, (t, (mean_t, cov_t)) in enumerate(zip(ts, snapshots)):
    shade = k / (len(ts) - 1)
    is_end = k in (0, len(ts) - 1)
    ax.add_patch(ellipse(mean_t, cov_t, edgecolor=cmap(shade), facecolor=cmap(shade) if is_end else "none",
                         alpha=0.35 if is_end else 1.0, linewidth=2.2, label=f"t = {t:.2f}"))
for p, q in zip(points, destinations):
    ax.plot([p[0], q[0]], [p[1], q[1]], color=viz.FLOW_COLOR, lw=1.2, alpha=0.9, zorder=3)
ax.scatter(points[:, 0], points[:, 1], s=26, color=viz.SOURCE_COLOR, edgecolor="white", zorder=4, label="points at t = 0")
ax.scatter(destinations[:, 0], destinations[:, 1], s=26, color=viz.TARGET_COLOR, edgecolor="white", zorder=4, label="the same points at t = 1")
ax.plot(means[:, 0], means[:, 1], "--", color=COLORS["text"], lw=2.4, label="mean trajectory")
ax.set_xlim(-6, 6)
ax.set_ylim(-5, 5)
ax.set_aspect("equal")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("The W2 geodesic between two Gaussians, with the tracks of individual points", pad=12)
ax.legend(loc="upper left", fontsize=9)
plt.show()

**Read the figure.** The ellipses (1.5 standard deviations) flow from the lower-left Gaussian (violet) to the upper-right one (yellow), with three outlines between them: the mean follows the dashed line while the shape turns and changes size. The green segments are the tracks of the fourteen points. At each time the cloud of points has the covariance of the ellipse of that time. The 1.5-standard-deviation contour holds only about two thirds of a Gaussian cloud, so a few dots fall outside their contour, at each end, as expected. Two of the 91 pairs of tracks cross in the picture, but never at the same moment: $x_t - y_t = M_t\,(x - y)$ cannot vanish because $M_t$ is invertible.

### Constant speed, and a rival path

Write $D = W_2(\mu_0, \mu_1)$. A path has **constant speed** along a shortest route when the distance from the start grows in proportion to the time, $W_2(\mu_0, \mu_t) = t\,D$, and its length (the sum of the distances between neighbouring snapshots) equals $D$. We check both on the covariances alone, with both means at zero.

We also set the geodesic beside a rival. The straight blend of the two covariance matrices,

$$ \Sigma_t^{\mathrm{blend}} = (1-t)\,\Sigma_0 + t\,\Sigma_1, $$

interpolates the entries of $\Sigma$ linearly, as a flat space would, and it stays inside the Gaussian family, since a mixture of positive matrices is positive. It is the straight line for the Euclidean metric on matrices, so the only question is whether it is shortest for $W_2$. The next cell measures, for both paths, the distance from the start divided by $D$ and the length. The right panel plots that ratio minus $t$, which is zero for constant speed.

In [ ]:
zero = np.zeros(2)
total = bures_wasserstein_distance(zero, cov_0, zero, cov_1)
grid = np.linspace(0.0, 1.0, 401)

geo_covs = [gaussian_geodesic(zero, cov_0, zero, cov_1, t)[1] for t in grid]
blend_covs = [(1 - t) * cov_0 + t * cov_1 for t in grid]

def from_start(covs):
    return np.array([bures_wasserstein_distance(zero, cov_0, zero, c) for c in covs]) / total

def length(covs):
    return sum(bures_wasserstein_distance(zero, a, zero, b) for a, b in zip(covs[:-1], covs[1:]))

ratio_geo, ratio_blend = from_start(geo_covs), from_start(blend_covs)
print(f"W2 between the two Gaussians (means removed): {total:.4f}")
print(f"length of the geodesic path:                  {length(geo_covs):.4f}")
print(f"length of the blend path:                     {length(blend_covs):.4f}")
for t in [0.25, 0.5, 0.75]:
    k = int(round(t * 400))
    print(f"t = {t:.2f}   distance from start / total:  geodesic {ratio_geo[k]:.4f}   blend {ratio_blend[k]:.4f}")
print(f"largest |ratio - t|: geodesic {np.abs(ratio_geo - grid).max():.1e}, blend {np.abs(ratio_blend - grid).max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(grid, ratio_geo, color=viz.FLOW_COLOR, lw=2.8, label="geodesic")
axes[0].plot(grid, ratio_blend, color=COLORS["highlight"], lw=2.4, label="covariance blend")
axes[0].plot(grid, grid, color=COLORS["muted"], lw=1, ls=":", label="constant speed")
axes[0].set_xlabel("time t")
axes[0].set_ylabel("W2(start, path at t) / W2(start, end)")
axes[0].set_title("Distance travelled from the start")
axes[0].legend(loc="upper left")
axes[1].plot(grid, ratio_geo - grid, color=viz.FLOW_COLOR, lw=2.8, label="geodesic")
axes[1].plot(grid, ratio_blend - grid, color=COLORS["highlight"], lw=2.4, label="covariance blend")
axes[1].axhline(0.0, color=COLORS["muted"], lw=1, ls=":")
axes[1].set_xlabel("time t")
axes[1].set_ylabel("distance ratio minus t")
axes[1].set_title("Departure from constant speed")
axes[1].legend(loc="lower left")
plt.tight_layout()
plt.show()

**Read the figure.** On the left the geodesic (green) lies on the dotted diagonal and the two curves are hard to tell apart; the right panel magnifies the gap. The geodesic departs from constant speed only by rounding error, while the blend falls below zero over most of the path: at half time it has covered only $0.4889$ of the distance. The geodesic has length $0.8470$, which is the distance itself, and the blend is longer, $0.8599$. The blend is a legitimate path of Gaussians that is not shortest for $W_2$.

## 4. A check on density matrices

A density matrix has unit trace, so a path of quantum states must keep the trace at $1$. Is the look-alike formula of the geodesic such a path?

**One more look: feed the geodesic density matrices.** In `03/13` the matrix term of $W_2$ became the Bures distance on density matrices. Does the geodesic formula $\Sigma_t = M_t\,\Sigma_0\,M_t$ become a path of quantum states in the same way? Its trace answers. Expanding $M_t = (1-t)I + tA$ gives $\mathrm{tr}\,\Sigma_t = (1-t)^2\,\mathrm{tr}\,\Sigma_0 + 2t(1-t)\,\mathrm{tr}(A\Sigma_0) + t^2\,\mathrm{tr}\,\Sigma_1$, using $A\Sigma_0A = \Sigma_1$ for the last term. By the formula for $A$, $\mathrm{tr}(A\Sigma_0) = \mathrm{tr}\sqrt{\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}}$, which for unit-trace inputs is the Uhlmann fidelity $F_U$ of `03/13`. Between two density matrices the trace is therefore $(1-t)^2 + 2t(1-t)\,F_U + t^2$: it equals $1$ at both ends and $(1 + F_U)/2$ at the midpoint, below $1$ whenever the two states differ. The next cell checks this on two qubit states.

In [ ]:
# Feed the Gaussian geodesic formula two density matrices (unit trace) and follow the trace.
rho_0 = np.array([[0.8, 0.1], [0.1, 0.2]])
rho_1 = np.array([[0.3, -0.2], [-0.2, 0.7]])
zero = np.zeros(2)
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    _, sigma_t = gaussian_geodesic(zero, rho_0, zero, rho_1, t)
    print(f"t = {t:.2f}   tr(Sigma_t) = {np.trace(sigma_t):.4f}")
fidelity = float(np.trace(gaussian_ot_map(rho_0, rho_1) @ rho_0))   # tr(A rho_0) = F_U
print(f"Uhlmann fidelity F_U = {fidelity:.4f},   (1 + F_U) / 2 = {(1 + fidelity) / 2:.4f}")

**Read the output.** The trace is $1$ at both ends and dips to $0.9060$ at the midpoint, which is $(1 + F_U)/2$ with $F_U = 0.8120$. Between its ends the curve leaves the set of density matrices. So the formula extends to density matrices, but the curve it draws is not a path of quantum states, and the Bures geodesic between two states, which stays among the states, is a different curve. This course names that geodesic and does not build it.

## 5. Dictionary update

This notebook adds rows to the classical$\leftrightarrow$quantum dictionary; `03/15` brings the last row of Module 3. Their quantum entries are of two kinds. The density matrix and the Bures distance you have already built, in `01/13` and `02/13`. The coupling map and the Bures geodesic are named for orientation and are not built in this course, as their rows say. For the geodesic row, the check of section 4 shows why the Gaussian formula cannot be carried over as it stands: fed two density matrices, it draws a curve that stays among positive matrices and fails only on the unit trace, which dips below $1$.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| covariance matrix $\Sigma$ (PSD, finite trace) | density matrix $\rho$ (PSD, unit trace) |
| Gaussian Bures–Wasserstein $W_2$ | Bures distance on density matrices (the bridge) |
| McCann map $T(x) = m_1 + A(x - m_0)$ | quantum coupling map (not built in this course) |
| $W_2$ geodesic on Gaussians | Bures geodesic on density matrices (implicit in `02/13` / Uhlmann; not built in this course) |

## Your turn

1. **A rotated pair.** Take $\Sigma_0 = \mathrm{diag}(4, 1)$ and $\Sigma_1 = R\,\mathrm{diag}(1, 9)\,R^\top$ for a rotation $R$ of your choice. Which statements of section 2 still hold for $A$: is it symmetric, is it positive, does $A\,\Sigma_0\,A = \Sigma_1$ hold? Check with `gaussian_ot_map`.
2. **A midpoint on the line, by hand.** Between $\mathcal{N}(1, 2^2)$ and $\mathcal{N}(7, 0.5^2)$, write the mean and standard deviation of the geodesic at $t = \tfrac12$ and $t = \tfrac14$ and check with `gaussian_geodesic`. What would interpolating the variance give instead?
3. **Commuting covariances.** When $\Sigma_0$ and $\Sigma_1$ commute, show that $A = \Sigma_0^{-1/2}\Sigma_1^{1/2}$ and that the covariance on the geodesic is $\bigl((1-t)\,\Sigma_0^{1/2} + t\,\Sigma_1^{1/2}\bigr)^2$: the square root of the covariance is interpolated linearly. Check it numerically with two non-diagonal covariances (use $Q\,\mathrm{diag}(\cdot)\,Q^\top$ with the same rotation $Q$ for both).
4. **The map from the start to any time is optimal (harder).** Show that the map $x \mapsto m_t + M_t\,(x - m_0)$ is the optimal map from $\mu_0$ to $\mu_t$, using Brenier's theorem as stated in section 2. Then compute $W_2(\mu_0, \mu_t)^2$ from the cost of that map and deduce $W_2(\mu_0, \mu_t) = t\,W_2(\mu_0, \mu_1)$. Hints: the cost of an affine map splits into a mean part and a matrix part, as in `03/12`; $M_t$ is symmetric positive definite, being a convex combination of $I$ and $A$.

## What you built

- You can work out the optimal map between one-dimensional Gaussians by hand and see that the geodesic interpolates the standard deviation.
- You can state Brenier's theorem as a cited result and use it to single out the symmetric positive map $A$, measured against a rival map at a visibly higher cost.
- You can trace the $W_2$ geodesic between two Gaussians in the plane and show by its length that it beats the straight blend of covariances.
- You can explain why the geodesic formula, fed two density matrices, draws a curve whose trace dips below $1$.
- You can add four rows to the classical$\leftrightarrow$quantum dictionary.

## What's next

`03/15_benamou_brenier_otto` looks at the same path as a flow, with a velocity for the mass at every point and every time, and shows that the squared distance is the least kinetic energy among all the flows that carry one distribution onto the other (Benamou and Brenier), which Otto's geometric reading turns into a geometry of the space of distributions.

## References

- Y. Brenier, "Polar factorization and monotone rearrangement of vector-valued functions", *Comm. Pure Appl. Math.* 44, 375–417 (1991). DOI:10.1002/cpa.3160440402
- R. McCann, "A convexity principle for interacting gases", *Adv. Math.* 128, 153–179 (1997). DOI:10.1006/aima.1997.1634
- R. Bhatia, T. Jain & Y. Lim, "On the Bures–Wasserstein distance between positive definite matrices", *Expo. Math.* 37, 165–191 (2019). DOI:10.1016/j.exmath.2018.01.002
- D. C. Dowson & B. V. Landau, "The Fréchet distance between multivariate normal distributions", *J. Multivar. Anal.* 12, 450–455 (1982). DOI:10.1016/0047-259X(82)90077-X
- I. Olkin & F. Pukelsheim, "The distance between two random vectors with given dispersion matrices", *Linear Algebra Appl.* 48, 257–263 (1982). DOI:10.1016/0024-3795(82)90112-4

The rival maps of section 2, the covariance blend of section 3 and the check on density matrices are the course's own illustrations.

**Previous:** `notebooks/03_ClassicalOptimalTransport/13_the_bures_bridge.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/15_benamou_brenier_otto.ipynb`